# Лабораторная работа №3: Описательные статистики

Освоить базовые методы **описательной статистики** и визуализации данных:

- расчёт мер центральной тенденции (среднее, медиана);
- расчёт мер разброса (стандартное отклонение, стандартная ошибка, квартили);
- оценка формы распределения (асимметрия, эксцесс);
- группировка данных (`groupby`) и сравнение подгрупп;
- построение графиков: гистограмма с KDE, boxplot, barplot.

---

## Описание датасета

Фрейм данных содержит **4406 наблюдений** по **19 переменным**. Для анализа используются следующие переменные:

| Переменная | Тип | Описание |
|-----------|-----|----------|
| `visits` | количественная | Количество посещений кабинета врача |
| `health` | категориальная | Самооценка здоровья: poor, average (эталон), excellent |
| `chronic` | количественная | Количество хронических состояний |
| `adl` | категориальная | Ограничение повседневной деятельности: limited / normal |
| `region` | категориальная | Регион: northeast, midwest, west, other |
| `age` | количественная | Возраст / 10 |
| `gender` | категориальная | Пол |
| `married` | категориальная | Семейное положение |
| `school` | количественная | Количество лет обучения |
| `income` | количественная | Доход в 10000 $ |
| `employed` | категориальная | Работает ли человек |
| `insurance` | категориальная | Частная страховка |

---

## Основные статистические термины

- **Среднее (mean)** — сумма всех значений, делённая на их количество. Чувствительно к выбросам.
- **Медиана (median)** — значение, делящее упорядоченную выборку пополам. Устойчива к выбросам (робастная оценка).
- **Стандартное отклонение (sd)** — мера разброса значений относительно среднего. Показывает, насколько в среднем значения отклоняются от среднего.
- **Стандартная ошибка среднего (SE)** — стандартное отклонение выборочного среднего. Показывает точность оценки среднего: `SE = sd / √n`.
- **Квартили (Q1, Q2, Q3)** — значения, делящие выборку на 4 равные части. Q2 = медиана. **IQR = Q3 − Q1** — межквартильный размах.
- **Асимметрия (skewness)** — мера скошенности распределения. > 0 — правый хвост длиннее, < 0 — левый хвост длиннее.
- **Эксцесс (kurtosis)** — мера «тяжести хвостов». > 0 — островершинное распределение с тяжёлыми хвостами, < 0 — плосковершинное.

---
## 0. Импорт библиотек и настройка окружения

Подключаем основные библиотеки для анализа данных и визуализации:

- **`os`** — работа с файловой системой (проверка наличия файла);
- **`numpy`** — численные операции (корень, массивы);
- **`pandas`** — таблицы данных (DataFrame) и группировка;
- **`scipy.stats`** — статистические функции (KDE, skew, kurtosis);
- **`matplotlib`** — базовые графики;
- **`seaborn`** — красивые статистические графики поверх matplotlib.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Настройка графиков по умолчанию
plt.rcParams["figure.figsize"] = (10, 6)   # размер фигуры
plt.rcParams["font.size"] = 12             # размер шрифта
sns.set_style("whitegrid")                  # белый фон с сеткой

---
## 1. Загрузка локального файла `NMES1988.csv`

Читаем CSV-файл из текущей рабочей директории.  
Параметр `index_col=0` убирает первый столбец с идентификаторами строк (`rownames`), чтобы он не мешал анализу.

In [ ]:
FILE_NAME = "NMES1988.csv"

# Проверяем, что файл существует рядом со скриптом/ноутбуком
if not os.path.exists(FILE_NAME):
    raise FileNotFoundError(
        f"Файл '{FILE_NAME}' не найден в {os.getcwd()}. "
        f"Положите файл рядом с ноутбуком."
    )

# Читаем CSV. index_col=0 => первый столбец (rownames) станет индексом
data = pd.read_csv(FILE_NAME, index_col=0)

print("Размерность исходного датасета:", data.shape)
print("Столбцы:", data.columns.tolist())
data.head()

**Что мы видим:**  
- Датасет содержит 4406 строк и 19 столбцов.  
- Каждая строка — один пациент.  
- Есть переменные разных типов: целочисленные счётчики (`visits`, `chronic`), вещественные (`age`, `income`, `school`), категориальные (`health`, `region`, `gender`, `married`, `adl`, `employed`, `insurance`).

---
## 2. Отбор нужных переменных и первичный осмотр

Оставляем только те столбцы, которые указаны в задании. Это упростит дальнейший анализ и избавит от лишнего шума.

In [ ]:
needed_cols = ["visits", "health", "chronic", "adl", "region",
               "age", "gender", "married", "school", "income",
               "employed", "insurance"]

# Проверяем, какие столбцы есть в датасете, а каких нет
available = [c for c in needed_cols if c in data.columns]
missing = [c for c in needed_cols if c not in data.columns]

print("Доступные столбцы:", available)
print("Отсутствующие столбцы:", missing if missing else "нет")

# Оставляем только нужные переменные
df = data[available].copy()
print("\nРазмерность после отбора:", df.shape)
df.head()

### 2.1. Проверка типов и пропусков

- `df.info()` показывает типы данных и количество непустых значений.  
- `df.isnull().sum()` показывает, сколько пропусков в каждом столбце.  

**Пропуски** — важный момент: если они есть, статистики могут считаться по разному числу наблюдений. В нашем датасете пропусков быть не должно.

In [ ]:
df.info()
print("\nПропуски по столбцам:")
print(df.isnull().sum())

### 2.2. Общая сводка по всем переменным (`describe(include='all')`)

Метод `describe()` возвращает основные статистики:

- для **количественных** переменных: `count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max`;
- для **категориальных**: `count`, `unique`, `top` (самое частое значение), `freq` (частота `top`).

In [ ]:
df.describe(include="all")

**Что видно из сводки:**  
- `visits` сильно варьируется: min = 0, max = 89, среднее ≈ 5.77.  
- `chronic` — от 0 до 8, медиана 1.  
- `age` — от 6.6 до 10.9 (в единицах «десятки лет»), т.е. примерно от 66 до 109 лет.  
- `income` — от отрицательных значений (долги/убытки) до ~54.  
- Категориальные: `health` — 3 уровня, `region` — 4, `gender` — 2 и т.д.

---
## 3. Гистограмма `visits` с кривой плотности (KDE)

**Гистограмма** — это график, который показывает, как часто значения попадают в тот или иной интервал (bin).  
**KDE (Kernel Density Estimation)** — сглаженная оценка плотности распределения. Помогает увидеть форму распределения без «ступенек» гистограммы.

Параметр `density=True` нормирует гистограмму так, чтобы её площадь была равна 1 — тогда KDE и гистограмма рисуются в одних единицах (плотность).

In [ ]:
fig, ax = plt.subplots()

# Гистограмма с нормировкой на плотность
ax.hist(df["visits"], bins=30, density=True, color="lightblue",
        edgecolor="black", alpha=0.7, label="гистограмма")

# KDE по данным
kde = stats.gaussian_kde(df["visits"].dropna())
xs = np.linspace(df["visits"].min(), df["visits"].max(), 300)
ax.plot(xs, kde(xs), color="crimson", lw=2, label="KDE")

ax.set_xlabel("visits (число посещений)")
ax.set_ylabel("плотность")
ax.set_title("Гистограмма visits + KDE")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# То же самое через seaborn — короче и красивее
plt.figure()
sns.histplot(df["visits"], bins=30, kde=True, color="steelblue",
             edgecolor="black", alpha=0.7)
plt.xlabel("visits")
plt.title("Гистограмма + KDE (seaborn)")
plt.tight_layout()
plt.show()

**Интерпретация:**  
Распределение `visits` сильно **асимметрично** с длинным правым хвостом.  
Большинство людей имеет 0–5 посещений, но есть редкие значения до 89.  
Это типично для счётчиковых данных (например, пуассоновское распределение с перерассеянием).

---
## 4. Среднее значение и стандартная ошибка (SE) для `visits`

**Стандартная ошибка среднего (SE)** показывает, насколько выборочное среднее может отличаться от истинного среднего в генеральной совокупности:

$$
\mathrm{SE} = \frac{s}{\sqrt{n}}
$$

где $s$ — выборочное стандартное отклонение, $n$ — объём выборки.

In [ ]:
x = df["visits"]
n = x.count()                       # количество наблюдений
mean_x = x.mean()                   # среднее
sd_x = x.std(ddof=1)                # стандартное отклонение (несмещённое)
se_x = sd_x / np.sqrt(n)            # стандартная ошибка среднего

print(f"n     = {n}")
print(f"mean  = {mean_x:.4f}")
print(f"sd    = {sd_x:.4f}")
print(f"SE    = {se_x:.4f}")

**Интерпретация:**  
- `mean ≈ 5.77` — в среднем человек посещает врача ~6 раз за период наблюдения.  
- `sd ≈ 7.87` — разброс большой: значения сильно отличаются от среднего.  
- `SE ≈ 0.12` — выборочное среднее отклоняется от истинного в среднем на 0.12 посещения. Это очень точная оценка благодаря большой выборке (n = 4406).

---
## 5. Медиана, квартили, summary для `visits`

- **Медиана** — 50-й процентиль, устойчивая к выбросам оценка центра.  
- **Квартили** — 25%, 50%, 75%. Разбивают выборку на 4 равные части.  
- **IQR = Q3 − Q1** — межквартильный размах, робастная мера разброса.

In [ ]:
print("Медиана:", x.median())
print("\nКвантили 0%, 25%, 50%, 75%, 100%:")
print(x.quantile([0, 0.25, 0.5, 0.75, 1.0]))
print()
print(x.describe())

**Интерпретация:**  
- Медиана = 5 меньше среднего = 5.77 — распределение скошено вправо.  
- IQR = Q3 − Q1 = 8 − 2 = 6 — половина людей имеет от 2 до 8 посещений.  
- Максимум = 89 — есть очень активные пациенты, которые сильно «тянут» среднее вверх.

---
## 6. Асимметрия (skewness) и эксцесс (kurtosis) для `visits`

**Skewness** (асимметрия):
- > 0 — правый хвост длиннее;  
- < 0 — левый хвост длиннее;  
- ≈ 0 — распределение симметрично.

**Kurtosis** (эксцесс, excess):
- > 0 — островершинное, тяжёлые хвосты;  
- < 0 — плосковершинное;  
- ≈ 0 — как у нормального распределения.

В `scipy.stats.kurtosis` по умолчанию считается **excess kurtosis** (нормальное = 0).

In [ ]:
sk = stats.skew(x, bias=False)         # несмещённая оценка
ku = stats.kurtosis(x, bias=False)     # excess kurtosis
print(f"skewness = {sk:.4f}")
print(f"kurtosis (excess) = {ku:.4f}")

**Интерпретация:**  
- Асимметрия ≈ 1.77 > 0 — выраженная правосторонняя асимметрия.  
- Эксцесс ≈ 4.42 > 0 — «тяжёлые хвосты», распределение островершинное.  
- Оба показателя подтверждают, что `visits` **не подчиняется нормальному закону**, что важно учитывать при выборе статистических тестов.

---
## 7. Медианы и квартили количественных данных по полу

Сравниваем распределения количественных переменных у мужчин и женщин.  
Медиана — устойчивая к выбросам мера центра, поэтому для счётчиковых данных она предпочтительнее среднего.

In [ ]:
quant_vars = ["visits", "chronic", "age", "school", "income"]

print("Медианы по полу:")
print(df.groupby("gender")[quant_vars].median().round(3))

In [ ]:
print("Квартили visits по полу:")
print(df.groupby("gender")["visits"]
        .quantile([0.25, 0.5, 0.75])
        .unstack()
        .round(3))

In [ ]:
# Полная сводка (count, mean, std, min, 25%, 50%, 75%, max) по каждому показателю
for var in quant_vars:
    print(f"\n--- {var} ---")
    print(df.groupby("gender")[var].describe().round(3))

**Интерпретация:**  
- Медиана `visits` у женщин (5) чуть выше, чем у мужчин (5).  
- По возрасту различия минимальны.  
- Медианный доход мужчин немного выше, чем у женщин.  
- Женщины в среднем чуть старше и чуть больше болеют хроническими заболеваниями.

---
## 8. Диаграмма размаха (boxplot) `visits` по полу

**Boxplot** (ящик с усами) — график, основанный на **робастных оценках**:

- **Медиана** — горизонтальная линия внутри ящика;  
- **Q1 и Q3** — нижняя и верхняя границы ящика;  
- **IQR = Q3 − Q1** — высота ящика;  
- **Усы** — до наибольшего/наименьшего значения в пределах 1.5 × IQR от границ ящика;  
- **Точки за усами** — потенциальные выбросы.

Параметр `whis` (или `range`) задаёт множитель IQR для усов. По умолчанию 1.5.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="gender", y="visits", palette="Set2")
plt.title("Количество посещений врача по полу")
plt.xlabel("Пол")
plt.ylabel("visits")
plt.tight_layout()
plt.show()

In [ ]:
# Сравнение двух значений параметра range (whis)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=df, x="gender", y="visits", ax=axes[0], palette="Set2")
axes[0].set_title("Boxplot (range=1.5, по умолчанию)")
axes[0].set_ylabel("visits")

sns.boxplot(data=df, x="gender", y="visits", ax=axes[1],
            palette="Set2", whis=3.0)
axes[1].set_title("Boxplot (range=3.0)")
axes[1].set_ylabel("visits")

plt.tight_layout()
plt.show()

**Интерпретация:**  
- Медиана и квартили у мужчин и женщин очень близки.  
- Оба распределения имеют длинный правый хвост и множество выбросов (люди с очень большим числом посещений).  
- При увеличении `whis` до 3.0 усы удлиняются, и меньше точек считаются выбросами — это иллюстрирует, что «выброс» — понятие относительное.

---
## 9. Медианы `visits` по `gender` и `region` одновременно

Сводная таблица: строки — регионы, столбцы — пол.  
Метод `.unstack()` превращает «второй уровень индекса» (пол) в столбцы — так удобнее сравнивать.

In [ ]:
med_two = df.groupby(["region", "gender"])["visits"].median().unstack()
print("Медианы visits: region × gender")
print(med_two.round(2))

In [ ]:
print("Полная сводка visits по region и gender:")
print(df.groupby(["region", "gender"])["visits"]
        .agg(["count", "mean", "median", "std"])
        .round(2))

**Интерпретация:**  
Медианное число посещений варьируется от 4 до 6 в зависимости от региона и пола.  
Наибольшие медианы — в регионе `other` у женщин (6), наименьшие — в `northeast` у мужчин (4).  
Различия между группами небольшие, но устойчивые.

---
## 10. Столбиковые диаграммы медиан `visits` по `gender` и `region`

Столбиковая диаграмма (barplot) — удобна для сравнения агрегатов (медиан, средних) между группами.

In [ ]:
med_two.plot(kind="bar", figsize=(9, 5), edgecolor="black",
             color=["#66c2a5", "#fc8d62"])
plt.ylabel("Медиана visits")
plt.xlabel("Регион")
plt.title("Медианное число посещений: регион × пол")
plt.xticks(rotation=0)
plt.legend(title="Пол")
plt.tight_layout()
plt.show()

In [ ]:
# Альтернатива через seaborn — сразу считает медиану внутри barplot
plt.figure(figsize=(9, 5))
sns.barplot(data=df, x="region", y="visits", hue="gender",
            estimator=np.median, errorbar=None,
            palette="Set2", edgecolor="black")
plt.title("Медианное число посещений: регион × пол (seaborn)")
plt.ylabel("Медиана visits")
plt.xlabel("Регион")
plt.tight_layout()
plt.show()

**Интерпретация:**  
Столбиковые диаграммы наглядно показывают, что **регион и пол слабо влияют** на медианное число посещений. Различия между группами невелики (4–6 посещений), но диаграмма помогает их увидеть.

---
## 11. Медианный возраст по `married`: вся выборка и по полу

Анализируем, как **семейное положение** связано с **возрастом**.  
Сначала считаем медианный возраст по всей выборке, затем — отдельно для мужчин и женщин.

In [ ]:
med_age_married = df.groupby("married")["age"].median()
print("Медианный возраст по married (вся выборка):")
print(med_age_married.round(2))

med_age_two = df.groupby(["married", "gender"])["age"].median().unstack()
print("\nМедианный возраст: married × gender")
print(med_age_two.round(2))

In [ ]:
# Barplot: медианный возраст по married для всей выборки
plt.figure(figsize=(7, 5))
med_age_married.plot(kind="bar", color="teal", edgecolor="black")
plt.ylabel("Медианный возраст (age)")
plt.xlabel("married")
plt.title("Медианный возраст по семейному положению")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Barplot: медианный возраст по married × gender
med_age_two.plot(kind="bar", figsize=(9, 5), edgecolor="black",
                 color=["#66c2a5", "#fc8d62"])
plt.ylabel("Медианный возраст (age)")
plt.xlabel("married")
plt.title("Медианный возраст: married × gender")
plt.xticks(rotation=0)
plt.legend(title="Пол")
plt.tight_layout()
plt.show()

In [ ]:
# Альтернатива через seaborn
plt.figure(figsize=(9, 5))
sns.barplot(data=df, x="married", y="age", hue="gender",
            estimator=np.median, errorbar=None,
            palette="Set2", edgecolor="black")
plt.title("Медианный возраст: married × gender (seaborn)")
plt.ylabel("Медианный возраст (age)")
plt.xlabel("married")
plt.tight_layout()
plt.show()

### Анализ результатов

**По всей выборке:**
- Медианный возраст женатых/замужних людей (≈ 7.4, т.е. ~74 года) выше, чем у одиноких (≈ 7.0, т.е. ~70 лет).
- Это ожидаемо: семейные люди, как правило, старше.

**По полу:**
- Разница в возрасте между женатыми и одинокими присутствует у обоих полов.
- У женщин медианный возраст и в браке, и вне брака чуть выше, чем у мужчин.

**Осторожная интерпретация:**
- Наблюдается **связь** между семейным положением и возрастом, но это **не означает причинно-следственную связь**.
- Возможны смещения: в выборке пожилых людей больше среди женатых.
- Для строгих выводов нужен регрессионный анализ с контролем на другие факторы (например, `health`, `chronic`).

---
## Итоговые выводы по лабораторной работе

1. **`visits`** — сильно асимметричная счётчиковая переменная с правосторонним хвостом. Среднее (5.77) > медианы (5).  
2. **Skewness ≈ 1.77**, **kurtosis ≈ 4.42** — распределение далеко от нормального.  
3. **Пол** слабо влияет на число посещений: медианы у мужчин и женщин близки.  
4. **Регион** также слабо влияет: медианы варьируются от 4 до 6.  
5. **Семейное положение** связано с возрастом: женатые/замужние в среднем старше.  
6. Для строгих выводов о влиянии факторов необходимы регрессионные модели (Пуассона / отрицательного биномиального распределения).

---

## Шпаргалка по методам

| Задача | Код |
|--------|-----|
| Загрузка CSV | `pd.read_csv("file.csv", index_col=0)` |
| Среднее / sd | `s.mean()`, `s.std(ddof=1)` |
| SE среднего | `s.std(ddof=1) / np.sqrt(s.count())` |
| Медиана, квантили | `s.median()`, `s.quantile([0.25, 0.5, 0.75])` |
| Summary | `s.describe()` / `df.describe(include="all")` |
| Skewness / kurtosis | `stats.skew(x)`, `stats.kurtosis(x)` |
| По группам | `df.groupby("A")["y"].median()` / `.describe()` |
| Два фактора | `df.groupby(["A", "B"])["y"].median().unstack()` |
| Гистограмма + KDE | `sns.histplot(x, kde=True)` |
| Boxplot | `sns.boxplot(data=df, x="A", y="y", whis=1.5)` |
| Barplot | `series.plot(kind="bar")` / `sns.barplot(...)` |